# Extended Grid — Colab GPU runtime for the heavy experiments

**Repository:** `TheFinix13/dissertation-project` (EEEM004 dissertation).

### Research question

**Does an AI trading agent that knows when it's uncertain protect a portfolio better than one that doesn't?**

This notebook runs the **full-scale version** of that experiment — testing our uncertainty-aware agent across 70 different stocks, with 10 independent training runs per stock, and much longer training (50,000 steps instead of 10,000). This is the evidence that the result isn't just a fluke on one stock.

### What this notebook does, step by step

1. **Setup** — clones the repo, installs dependencies, checks the GPU is working.
2. **Smoke test** — a quick 2-minute test on one stock to make sure everything runs.
3. **The main experiment** — trains all four strategies (buy-and-hold, manual stop-loss, baseline AI, uncertainty-aware AI) on **70 diversified stocks × 10 random seeds × 50,000 training steps**. This is ~5–7 hours on a T4 GPU.
4. **Walk-forward test** — checks if the agent works on time periods it's never seen before (out-of-sample validation).
5. **Results** — aggregates everything and rebuilds the dissertation document with the new numbers.

### Key terms

| Term | Plain English |
|---|---|
| **broad-universe universe** | 70 real US stocks and ETFs — a diverse mix of tech, finance, healthcare, and broad market funds |
| **10 seeds** | Each stock is trained 10 times with different random starting points, so results reflect the method, not luck |
| **50,000 timesteps** | The AI practises trading for 50,000 simulated days per training run (much more than the quick 10,000-step test) |
| **Walk-forward** | Train on old data, test on newer data the AI has never seen — the gold standard for proving it actually works |
| **Bootstrap** | A statistical technique that creates synthetic variations of the training data to make the AI more robust |
| **Sharpe ratio** | Return per unit of risk. Higher is better. Above 0.5 is decent, above 1.0 is excellent |
| **Max drawdown (MDD)** | Worst peak-to-trough portfolio drop. Lower is better. Our target: keep drawdown below buy-and-hold |
| **PPO** | Proximal Policy Optimization — the algorithm the AI uses to learn. Standard in reinforcement learning |

### Runtime

**GPU required.** *Runtime → Change runtime type → Hardware accelerator: T4 GPU* (A100 if you have Pro+). Then *Runtime → Run all*.

> This notebook is for the **heavy experiments only** (~5–7 hours on T4). For a quick interactive walkthrough that runs on CPU, use `Dissertation_Walkthrough.ipynb` instead.


## 1 · Setup — clone, install, GPU sanity

Run the next cell once per Colab session. It is idempotent: if the repo is already cloned, `git pull` brings it up to date; if the dependencies are already installed, `pip install` is a no-op.

In [ ]:
import os, subprocess, sys, shlex, time, pathlib, json

REPO_URL  = "https://github.com/the1finix/dissertation-project.git"
REPO_NAME = "dissertation-project"
WORKDIR   = pathlib.Path("/content") / REPO_NAME

def sh(cmd, cwd=None, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, cwd=cwd, shell=True, text=True, capture_output=False)
    if check and r.returncode != 0:
        raise SystemExit(f"command failed: {cmd}")
    return r.returncode

if not WORKDIR.exists():
    sh(f"git clone --depth 1 {REPO_URL} {WORKDIR}")
else:
    sh("git pull --ff-only", cwd=str(WORKDIR), check=False)

os.chdir(WORKDIR)
sys.path.insert(0, str(WORKDIR))
sh("pip install -q -r requirements.txt")

import torch
print(f"\nPython:        {sys.version.split()[0]}")
print(f"PyTorch:       {torch.__version__}")
print(f"CUDA built:    {torch.version.cuda}")
print(f"CUDA available:{torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device:        {torch.cuda.get_device_name(0)}")
    print(f"Mem (GiB):     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}")
else:
    print("WARNING — no GPU detected. Switch the runtime to T4 / A100 or this notebook will be useless.")


## 2 · GPU smoke test — one ticker, one seed, 5 000 timesteps

Verifies that the probabilistic agent actually moves data through the GPU before launching anything heavy. Should finish in ~1 minute on a T4.

In [ ]:
t0 = time.time()
sh(
    "python experiments/runners/run_probabilistic_agent.py "
    "--tickers SPY --seeds 7 --timesteps 5000 --tag _gpu_smoke"
)
print(f"\nGPU smoke test finished in {time.time() - t0:.0f} s.")


**What just happened:** We trained the AI on a single stock (SPY) for a very short session (5,000 steps) just to confirm that the GPU, the code, and all dependencies are working correctly. If this cell finished without errors, we're ready for the real experiment.


## 3 · The main experiment — 70 diversified stocks × 10 seeds × 50,000 steps

This is the experiment that produces the dissertation's headline results. A single command trains and evaluates **all four strategies** on every stock:

- **Buy-and-hold** — just buy the stock and hold it. The simplest benchmark.
- **Manual 5% stop-loss** — a traditional rule: sell if the price drops 5% from its peak, buy back when a moving average signals recovery.
- **Baseline PPO** — an AI agent that trades without any uncertainty signal.
- **Probabilistic PPO** — our uncertainty-aware AI agent (the one we're testing).

Each stock is trained 10 times (10 seeds) to ensure consistency. The bootstrap augmentation creates 32 synthetic variations of the training data for each run, making the AI more robust.

**Expected runtime: 5–7 hours on T4, ~2 hours on A100.**


In [ ]:
t0 = time.time()
sh(
    "python experiments/runners/run_extended_grid.py "
    "--tickers broad_universe "
    "--seeds extended "
    "--timesteps 50000 "
    "--bootstrap-paths 32 "
    "--tag colab70 "
    "--skip-walk-forward"
)
elapsed = time.time() - t0
print(f"\nbroad-universe extended grid finished in {elapsed/3600:.2f} h "
      f"({elapsed/60:.1f} min, {elapsed:.0f} s).")


**What just happened:** The AI was trained and evaluated on all 70 diversified stocks. For each stock, we now have metrics (Sharpe ratio, max drawdown, final portfolio value, etc.) averaged across 10 independent training runs. These results files are saved in `experiments/results/` and will be picked up by the document builder in Section 6.


## 4 · Walk-forward test — proving it works on unseen data

The main experiment above trains on 2009–2018 and tests on 2022–2025. But a sceptic might ask: *"Maybe the AI just memorised something specific to those dates?"*

The walk-forward test answers this. It trains and tests on **four different time periods**:

| Fold | Train on | Test on | What it captures |
|---|---|---|---|
| 1 | 2009–2017 | 2018–2019 | Late-cycle bull market |
| 2 | 2009–2019 | 2020–2021 | COVID crash + recovery |
| 3 | 2009–2021 | 2022–2023 | 2022 bear market + rate hikes |
| 4 | 2009–2023 | 2024–2025 | Most recent window |

If the uncertainty-aware agent beats the baseline across multiple folds, we can be much more confident the result is real.

**Expected runtime: 3–4 hours on T4, ~1.5 hours on A100.**


In [ ]:
t0 = time.time()
sh(
    "python experiments/runners/run_walk_forward.py "
    "--tickers basket "
    "--folds all "
    "--seeds extended "
    "--timesteps 50000 "
    "--bootstrap-paths 16 "
    "--agents baseline,probabilistic "
    "--tag colab_wf_basket"
)
elapsed = time.time() - t0
print(f"\nBasket walk-forward grid finished in {elapsed/3600:.2f} h "
      f"({elapsed/60:.1f} min, {elapsed:.0f} s).")


## 5 · *(Optional, A100 only)* — full broad-universe walk-forward

The most extensive single experiment in the project: 70 diversified stocks × 4 folds × 10 seeds × 50 000 timesteps × 16 bootstrap paths × two agents = ~5 600 individual trainings. Do **not** run on a T4 — the session will time out. Enable only on A100 / V100, where it takes ~12–14 hours.

`RUN_FULL_70_WF = True` flips it on.

In [ ]:
RUN_FULL_70_WF = False  # flip to True only on A100 / V100

if RUN_FULL_70_WF:
    t0 = time.time()
    sh(
        "python experiments/runners/run_walk_forward.py "
        "--tickers broad_universe "
        "--folds all "
        "--seeds extended "
        "--timesteps 50000 "
        "--bootstrap-paths 16 "
        "--agents baseline,probabilistic "
        "--tag colab_wf_broad"
    )
    elapsed = time.time() - t0
    print(f"\nFull broad-universe walk-forward finished in {elapsed/3600:.2f} h.")
else:
    print("Skipped — set RUN_FULL_70_WF = True on an A100/V100 runtime to enable.")


## 6 · Aggregate, rebuild documents, generate charts

Pulls the new result JSONs from `experiments/results/`, aggregates them, and rebuilds `Main_Dissertation_Draft.docx` with the extended-budget numbers folded in.

In [ ]:
sh("python experiments/aggregate_results.py --tag colab70")
sh("python reports/builders/build_main_dissertation_docx.py")
sh("python reports/builders/build_interim_review_docx.py")

print("\nGenerated documents:")
for p in sorted(pathlib.Path("reports/generated/exports").glob("*.docx")):
    print(f"  {p.relative_to(WORKDIR)}  ({p.stat().st_size/1024:.0f} KiB)")
print("\nGenerated charts:")
for p in sorted(pathlib.Path("reports/generated/charts").glob("*.png")):
    print(f"  {p.relative_to(WORKDIR)}  ({p.stat().st_size/1024:.0f} KiB)")


## 7 · Results summary — what did we find?

The cell below reads the experiment results and prints a summary table. Here's how to interpret the key numbers:

- **Mean Sharpe > 0** — the strategy made money after accounting for risk. Higher is better.
- **Mean MDD** — average worst drawdown across all 70 diversified stocks. Lower is better. Our target: beat buy-and-hold.
- **MDD < B&H on X/70** — on how many stocks did our agent have a smaller drawdown than simply holding? We want this close to 70/70.
- **Final > B&H on X/70** — on how many stocks did our agent end up with more money than buy-and-hold? This is harder to achieve because reducing drawdown often costs some upside.


In [ ]:
import json, statistics as st
from pathlib import Path

results = Path("experiments/results")

def latest(prefix, tag):
    files = sorted(results.glob(f"{prefix}_*_{tag}.json"))
    return json.loads(files[-1].read_text()) if files else []

bench = latest("benchmarks", "colab70")
rule  = latest("rule_baseline", "colab70")
base  = latest("baseline", "colab70")
prob  = latest("probabilistic", "colab70")

def mean_metric(rows, key, agent_filter=None):
    if agent_filter:
        rows = [r for r in rows if r.get("agent") == agent_filter]
    vals = [float(r[key]) for r in rows if r.get(key) is not None]
    return sum(vals) / len(vals) if vals else float("nan")

bh   = [r for r in bench if r.get("agent") == "buy_and_hold"]
stop = [r for r in rule  if r.get("agent") == "stop_loss_5pct"]

print("=== broad-universe extended grid (tag = colab70) — aggregate ===")
print(f"  Buy-and-hold       final={mean_metric(bh,   'final_portfolio_value'):>14,.0f}   sharpe={mean_metric(bh,   'sharpe_ratio'):+.2f}   mdd={mean_metric(bh,   'max_drawdown'):.3f}")
print(f"  Manual 5% stop     final={mean_metric(stop, 'final_portfolio_value'):>14,.0f}   sharpe={mean_metric(stop, 'sharpe_ratio'):+.2f}   mdd={mean_metric(stop, 'max_drawdown'):.3f}")
print(f"  Baseline PPO       final={mean_metric(base, 'final_portfolio_value'):>14,.0f}   sharpe={mean_metric(base, 'sharpe_ratio'):+.2f}")
print(f"  Probabilistic PPO  final={mean_metric(prob, 'final_portfolio_value'):>14,.0f}   sharpe={mean_metric(prob, 'sharpe_ratio'):+.2f}   mdd={mean_metric(prob, 'max_drawdown'):.3f}")

bh_by_t = {r["ticker"]: r for r in bh}
prob_by_t: dict[str, list[float]] = {}
for r in prob:
    prob_by_t.setdefault(r["ticker"], []).append(float(r["max_drawdown"]))
dd_wins = sum(
    1 for t, mdds in prob_by_t.items()
    if t in bh_by_t and st.median(mdds) < float(bh_by_t[t]["max_drawdown"])
)
print(f"\nDrawdown reduction vs B&H: {dd_wins} of {len(prob_by_t)} tickers")


**How to read the output above:** The most important line is the probabilistic PPO row. If it shows:
- MDD < B&H on **70/70** stocks — the uncertainty-aware agent reduced drawdown on *every single stock* vs buy-and-hold
- A positive Sharpe ratio — it made risk-adjusted money, not just preserved capital
- Final value close to or above buy-and-hold on a good portion of stocks — it didn't sacrifice too much return for the protection

This is the core evidence for the dissertation: uncertainty-awareness delivers systematic drawdown reduction across a diverse stock universe.


## 8 · Package outputs — one zip, two delivery options

* **Option A · Direct download.** Smallest convenience: zip everything important and trigger a browser download.
* **Option B · Google Drive mount.** For multi-day grids on a Pro+ runtime: copy the zip to `/MyDrive/dissertation-project/` so the next session can resume from disk.

In [ ]:
import shutil, datetime

stamp = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
bundle_name = f"portfolio_risk_drl_extended_{stamp}"
bundle_dir = pathlib.Path("/content") / bundle_name
bundle_dir.mkdir(exist_ok=True)

for src in [
    "experiments/results",
    "reports/generated/exports",
    "reports/generated/charts",
    "reports/generated/figures",
]:
    p = pathlib.Path(src)
    if p.exists():
        shutil.copytree(p, bundle_dir / p.name, dirs_exist_ok=True)

archive = shutil.make_archive(str(bundle_dir), "zip", bundle_dir)
size_mb = pathlib.Path(archive).stat().st_size / 1e6
print(f"\nBundle: {archive}  ({size_mb:.1f} MiB)")

# --- Option A: direct download
try:
    from google.colab import files
    files.download(archive)
except Exception as exc:
    print(f"(skipping browser download: {exc})")

# --- Option B: copy to Drive (uncomment to enable)
# from google.colab import drive
# drive.mount("/content/drive", force_remount=False)
# drive_target = pathlib.Path("/content/drive/MyDrive/dissertation-project")
# drive_target.mkdir(parents=True, exist_ok=True)
# shutil.copy2(archive, drive_target / pathlib.Path(archive).name)
# print(f"Copied to {drive_target / pathlib.Path(archive).name}")


---

## What this notebook does *not* do (and why)

Everything that the project's CPU pipeline already handles cheaply is intentionally **left to CPU** — the smoke-tests, the eight-ticker basket at the Phase-1 budget, the four-ticker × four-fold walk-forward grid at 10 000 timesteps, the document builds, the supervisor-pack zip. They do not benefit from a GPU and they would just consume Colab quota.

Conversely, the items below were **specifically pushed off CPU and into this notebook** because the CPU runtime makes them either painfully slow or outright infeasible:

| Experiment                                                    | CPU runtime           | Colab T4 runtime | Notebook cell |
| ---                                                           | ---                   | ---              | --- |
| broad-universe × 10-seed × 50 000-step extended grid + bootstrap   | 2–3 days              | 5–7 hours        | Section 3 |
| 8-ticker × 4-fold × 10-seed × 50 000-step walk-forward        | 8–12 hours            | 3–4 hours        | Section 4 |
| Full broad-universe walk-forward (A100 only)                       | infeasible            | 12–14 h on A100  | Section 5 |
| Document rebuild with extended numbers folded in              | (depends on grid)     | < 1 minute       | Section 6 |

If a future experiment is heavier than the eight-ticker basket at Phase-1 budget, add it here rather than to the CPU pipeline.